# CricketPulse — Win Probability Model Training
**Author:** Himanshu Singh | **Event:** GDG Raipur Hackathon | **May 24, 2026**

Train an XGBoost win probability model on Kaggle IPL data.
Output: `models/win_prob.pkl` + `models/team_encoder.pkl`

**Expected runtime:** < 2 minutes total

In [ ]:
# Install dependencies if needed
# !pip install xgboost scikit-learn pandas numpy joblib

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)
print('Libraries loaded OK')

Libraries loaded OK


## Step 1 — Load Data
Put `deliveries.csv` and `matches.csv` in the same folder as this notebook.

In [2]:
deliveries = pd.read_csv('deliveries.csv')
matches    = pd.read_csv('matches.csv')

print(f'Deliveries shape : {deliveries.shape}')
print(f'Matches shape    : {matches.shape}')
print()
print('Deliveries columns:', deliveries.columns.tolist())
print('Matches columns   :', matches.columns.tolist())

Deliveries shape : (260920, 17)
Matches shape    : (1095, 20)

Deliveries columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']
Matches columns   : ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']


## Step 2 — Build Winner Column
Map each match_id to whether the batting team won (1) or lost (0).

In [3]:
# Keep only completed matches with a winner
matches_clean = matches[matches['winner'].notna() & (matches['winner'] != '')].copy()
print(f'Completed matches: {len(matches_clean)}')

# Map match_id → winner team name
winner_map = matches_clean.set_index('id')['winner'].to_dict()

# Filter deliveries to only those matches
df = deliveries[deliveries['match_id'].isin(winner_map.keys())].copy()
print(f'Deliveries after filter: {len(df)}')

# Add winner column
df['winner'] = df['match_id'].map(winner_map)

# Target: did the batting team win?
df['batting_team_won'] = (df['batting_team'] == df['winner']).astype(int)

print(f"\nTarget distribution:")
print(df['batting_team_won'].value_counts())

Completed matches: 1090
Deliveries after filter: 260430

Target distribution:
batting_team_won
0    132945
1    127485
Name: count, dtype: int64


## Step 3 — Compute Cumulative Match State Per Ball
For each ball, compute the running match state at that moment.

In [4]:
# Sort for correct cumulative computation
df = df.sort_values(['match_id', 'inning', 'over', 'ball']).reset_index(drop=True)

# Cumulative runs and wickets per match per inning
df['runs_so_far']     = df.groupby(['match_id', 'inning'])['total_runs'].cumsum()
df['wickets_so_far']  = df.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

# Balls bowled in this inning so far (1-indexed)
df['ball_number'] = df.groupby(['match_id', 'inning']).cumcount() + 1
df['balls_remaining'] = np.maximum(120 - df['ball_number'], 0)

# Overs elapsed (fractional)
df['overs_elapsed'] = df['ball_number'] / 6.0

# Current run rate (avoid divide by zero)
df['current_rr'] = np.where(
    df['overs_elapsed'] > 0,
    df['runs_so_far'] / df['overs_elapsed'],
    0.0
).clip(0, 36)

print('Cumulative features built')
print(df[['match_id','inning','over','ball','runs_so_far','wickets_so_far','ball_number','current_rr']].head(10))

Cumulative features built
   match_id  inning  over  ball  runs_so_far  wickets_so_far  ball_number  \
0    335982       1     0     1            1               0            1   
1    335982       1     0     2            1               0            2   
2    335982       1     0     3            2               0            3   
3    335982       1     0     4            2               0            4   
4    335982       1     0     5            2               0            5   
5    335982       1     0     6            2               0            6   
6    335982       1     0     7            3               0            7   
7    335982       1     1     1            3               0            8   
8    335982       1     1     2            7               0            9   
9    335982       1     1     3           11               0           10   

   current_rr  
0    6.000000  
1    3.000000  
2    4.000000  
3    3.000000  
4    2.400000  
5    2.000000  
6    2.571429 

## Step 4 — Add 2nd Innings Target Features
For 2nd innings: required run rate and runs remaining.

In [5]:
# Get total 1st innings score per match
first_innings = df[df['inning'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
first_innings.columns = ['match_id', 'innings1_total']

df = df.merge(first_innings, on='match_id', how='left')

# Target for 2nd innings = 1st innings total + 1
df['target'] = np.where(df['inning'] == 2, df['innings1_total'] + 1, 0)

# Runs remaining
df['runs_remaining'] = np.where(
    df['inning'] == 2,
    np.maximum(df['target'] - df['runs_so_far'], 0),
    0
)

# Required run rate
df['required_rr'] = np.where(
    (df['inning'] == 2) & (df['balls_remaining'] > 0),
    (df['runs_remaining'] / (df['balls_remaining'] / 6.0)).clip(0, 36),
    0
)

print('2nd innings features added')
df[df['inning']==2][['match_id','over','runs_so_far','target','runs_remaining','required_rr']].head(8)

2nd innings features added


,match_id,over,runs_so_far,target,runs_remaining,required_rr
124,335982,0,1,223,222,11.193277
125,335982,0,2,223,221,11.237288
126,335982,0,2,223,221,11.333333
127,335982,0,3,223,220,11.379310
128,335982,0,4,223,219,11.426087
129,335982,0,4,223,219,11.526316
130,335982,0,4,223,219,11.628319
131,335982,1,4,223,219,11.732143


## Step 5 — Encode Teams

In [6]:
le_batting  = LabelEncoder()
le_bowling  = LabelEncoder()

# Fit on ALL teams seen across both columns
all_teams = pd.concat([df['batting_team'], df['bowling_team']]).unique()
le_batting.fit(all_teams)
le_bowling.fit(all_teams)

df['batting_team_enc'] = le_batting.transform(df['batting_team'])
df['bowling_team_enc'] = le_bowling.transform(df['bowling_team'])

print(f'Teams found: {len(all_teams)}')
print(all_teams)

Teams found: 19
<StringArray>
[      'Kolkata Knight Riders', 'Royal Challengers Bangalore',
         'Chennai Super Kings',             'Kings XI Punjab',
            'Rajasthan Royals',            'Delhi Daredevils',
              'Mumbai Indians',             'Deccan Chargers',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',     'Rising Pune Supergiants',
               'Gujarat Lions',      'Rising Pune Supergiant',
              'Delhi Capitals',                'Punjab Kings',
        'Lucknow Super Giants',              'Gujarat Titans',
 'Royal Challengers Bengaluru']
Length: 19, dtype: str


## Step 6 — Build Feature Matrix

In [7]:
FEATURES = [
    'inning',
    'over',
    'ball',
    'runs_so_far',
    'wickets_so_far',
    'balls_remaining',
    'current_rr',
    'required_rr',
    'runs_remaining',
    'target',
    'batting_team_enc',
    'bowling_team_enc',
]

TARGET = 'batting_team_won'

# Drop rows with any NaN in features
model_df = df[FEATURES + [TARGET]].dropna()

X = model_df[FEATURES].values
y = model_df[TARGET].values

print(f'Training samples : {len(X):,}')
print(f'Feature count    : {len(FEATURES)}')
print(f'Class balance    : {y.mean():.3f} (batting team wins)')

Training samples : 260,430
Feature count    : 12
Class balance    : 0.490 (batting team wins)


## Step 7 — Train XGBoost

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print('Training XGBoost...')

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print('Training complete!')

Train: 208,344 | Test: 52,086
Training XGBoost...
[0]	validation_0-logloss:0.68282
[50]	validation_0-logloss:0.52739
[100]	validation_0-logloss:0.49350
[150]	validation_0-logloss:0.46123
[200]	validation_0-logloss:0.43407
[250]	validation_0-logloss:0.41275
[299]	validation_0-logloss:0.39715
Training complete!


## Step 8 — Evaluate

In [9]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc     = accuracy_score(y_test, y_pred)
auc     = roc_auc_score(y_test, y_pred_prob)
logloss = log_loss(y_test, y_pred_prob)

print('=' * 40)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  ROC-AUC   : {auc:.4f}')
print(f'  Log Loss  : {logloss:.4f}')
print('=' * 40)

# Quick sanity: feature importances
import pandas as pd
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nTop feature importances:')
print(fi.to_string())

  Accuracy  : 0.8169  (81.7%)
  ROC-AUC   : 0.9104
  Log Loss  : 0.3971

Top feature importances:
required_rr         0.252541
wickets_so_far      0.118325
runs_remaining      0.095644
target              0.090428
inning              0.077432
current_rr          0.073042
runs_so_far         0.070034
batting_team_enc    0.067033
bowling_team_enc    0.066445
balls_remaining     0.040501
over                0.040440
ball                0.008135


## Step 9 — Save Model + Encoder + Feature List

In [10]:
joblib.dump(model,      '../models/win_prob.pkl')
joblib.dump(le_batting, '../models/team_encoder.pkl')
joblib.dump(FEATURES,   '../models/features.pkl')

print('Saved:')
print('  ../models/win_prob.pkl      — XGBoost model')
print('  ../models/team_encoder.pkl  — LabelEncoder for teams')
print('  ../models/features.pkl      — feature name list')

Saved:
  ../models/win_prob.pkl      — XGBoost model
  ../models/team_encoder.pkl  — LabelEncoder for teams
  ../models/features.pkl      — feature name list


## Step 10 — Test Inference (What the Agent Will Do Per Ball)

In [11]:
# Simulate what win_probability.py does at runtime
loaded_model   = joblib.load('../models/win_prob.pkl')
loaded_encoder = joblib.load('../models/team_encoder.pkl')
loaded_features = joblib.load('../models/features.pkl')

def predict_win_probability(match_state: dict) -> float:
    """
    match_state keys:
        inning, over, ball, runs_so_far, wickets_so_far,
        balls_remaining, current_rr, required_rr,
        runs_remaining, target, batting_team, bowling_team
    Returns: float (0-100), batting team win probability
    """
    try:
        batting_enc = loaded_encoder.transform([match_state['batting_team']])[0]
        bowling_enc = loaded_encoder.transform([match_state['bowling_team']])[0]
    except ValueError:
        batting_enc = 0
        bowling_enc = 0

    features = [
        match_state.get('inning', 1),
        match_state.get('over', 0),
        match_state.get('ball', 0),
        match_state.get('runs_so_far', 0),
        match_state.get('wickets_so_far', 0),
        match_state.get('balls_remaining', 120),
        match_state.get('current_rr', 0.0),
        match_state.get('required_rr', 0.0),
        match_state.get('runs_remaining', 0),
        match_state.get('target', 0),
        batting_enc,
        bowling_enc,
    ]

    prob = loaded_model.predict_proba([features])[0][1]
    return round(float(prob) * 100, 1)


# --- Test cases ---

# 1st innings, over 5, MI scoring well
state_1 = {
    'inning': 1, 'over': 5, 'ball': 3,
    'runs_so_far': 52, 'wickets_so_far': 0,
    'balls_remaining': 87, 'current_rr': 8.9,
    'required_rr': 0.0, 'runs_remaining': 0, 'target': 0,
    'batting_team': 'Mumbai Indians',
    'bowling_team': 'Kolkata Knight Riders'
}

# 2nd innings, needing 40 off 18 balls, 2 wickets down
state_2 = {
    'inning': 2, 'over': 17, 'ball': 0,
    'runs_so_far': 140, 'wickets_so_far': 2,
    'balls_remaining': 18, 'current_rr': 7.7,
    'required_rr': 13.3, 'runs_remaining': 40, 'target': 180,
    'batting_team': 'Mumbai Indians',
    'bowling_team': 'Kolkata Knight Riders'
}

# 2nd innings, last over, need 6 off 6
state_3 = {
    'inning': 2, 'over': 19, 'ball': 0,
    'runs_so_far': 174, 'wickets_so_far': 4,
    'balls_remaining': 6, 'current_rr': 8.7,
    'required_rr': 6.0, 'runs_remaining': 6, 'target': 180,
    'batting_team': 'Mumbai Indians',
    'bowling_team': 'Kolkata Knight Riders'
}

p1 = predict_win_probability(state_1)
p2 = predict_win_probability(state_2)
p3 = predict_win_probability(state_3)

print('Win Probability Test Cases')
print('=' * 45)
print(f'1st innings, over 5, 52/0 (MI batting)  → MI: {p1:.1f}%')
print(f'2nd inn, need 40 off 18, 2 wkts down    → MI: {p2:.1f}%')
print(f'2nd inn, need 6 off 6, 4 wkts down      → MI: {p3:.1f}%')
print('=' * 45)
print()
print('Model is ready for CricketPulse!')
print('Commit models/win_prob.pkl before the hackathon.')

Win Probability Test Cases
1st innings, over 5, 52/0 (MI batting)  → MI: 69.6%
2nd inn, need 40 off 18, 2 wkts down    → MI: 91.9%
2nd inn, need 6 off 6, 4 wkts down      → MI: 98.9%

Model is ready for CricketPulse!
Commit models/win_prob.pkl before the hackathon.


## Step 11 — Generate the `win_probability.py` Agent Code

In [12]:
agent_code = '''
# app/agents/win_probability.py
import joblib
import numpy as np
from pathlib import Path

MODEL_PATH   = Path(__file__).parent.parent.parent / "models" / "win_prob.pkl"
ENCODER_PATH = Path(__file__).parent.parent.parent / "models" / "team_encoder.pkl"

_model   = None
_encoder = None

def load_model():
    """Call once at FastAPI startup via lifespan."""
    global _model, _encoder
    _model   = joblib.load(MODEL_PATH)
    _encoder = joblib.load(ENCODER_PATH)
    print(f"[WinProbAgent] Model loaded from {MODEL_PATH}")

def predict_win_probability(match_state: dict) -> float:
    """
    Returns batting team win probability as a percentage (0-100).
    Called on every ball event — runs in microseconds.
    """
    if _model is None:
        return 50.0  # fallback before model loads

    try:
        batting_enc = _encoder.transform([match_state["batting_team"]])[0]
        bowling_enc = _encoder.transform([match_state["bowling_team"]])[0]
    except (ValueError, KeyError):
        batting_enc = 0
        bowling_enc = 0

    features = [
        match_state.get("inning", 1),
        match_state.get("over", 0),
        match_state.get("ball", 0),
        match_state.get("runs_so_far", 0),
        match_state.get("wickets_so_far", 0),
        match_state.get("balls_remaining", 120),
        match_state.get("current_rr", 0.0),
        match_state.get("required_rr", 0.0),
        match_state.get("runs_remaining", 0),
        match_state.get("target", 0),
        batting_enc,
        bowling_enc,
    ]

    prob = _model.predict_proba([features])[0][1]
    return round(float(prob) * 100, 1)
'''

print(agent_code)
print()
print('Copy the above into app/agents/win_probability.py')


# app/agents/win_probability.py
import joblib
import numpy as np
from pathlib import Path

MODEL_PATH   = Path(__file__).parent.parent.parent / "models" / "win_prob.pkl"
ENCODER_PATH = Path(__file__).parent.parent.parent / "models" / "team_encoder.pkl"

_model   = None
_encoder = None

def load_model():
    """Call once at FastAPI startup via lifespan."""
    global _model, _encoder
    _model   = joblib.load(MODEL_PATH)
    _encoder = joblib.load(ENCODER_PATH)
    print(f"[WinProbAgent] Model loaded from {MODEL_PATH}")

def predict_win_probability(match_state: dict) -> float:
    """
    Returns batting team win probability as a percentage (0-100).
    Called on every ball event — runs in microseconds.
    """
    if _model is None:
        return 50.0  # fallback before model loads

    try:
        batting_enc = _encoder.transform([match_state["batting_team"]])[0]
        bowling_enc = _encoder.transform([match_state["bowling_team"]])[0]
    except (ValueError, KeyError):
        b